# 1. praktika: oinarrizko eredua probatu eta oinarri-lerroa neurtu

**Helburua**: `Qwen2.5-0.5B-Instruct` eredua kargatu, harekin hitz egin eta, **ezer entrenatu aurretik**, IKT-txartelen sailkapen-atazan zein ondo dabilen neurtu (zero-shot eta few-shot).

**Aurretik**: *Exekuzio-ingurunea → Aldatu exekuzio-ingurune mota → T4 GPU*.

Emaitzak `oinarri_lerroa.csv` fitxategian gordeko dira (4. saioan behar dira).

## 1. Ingurunea prestatu

In [ ]:
# COLAB: liburutegiak instalatu (bertsio finkoak, unitate osoan berdinak)
!pip install -q transformers==5.22.2 peft==0.21.2 trl==1.17.1 datasets==5.2.0

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# COLAB: Google Drive muntatu (emaitzak gordetzeko eta datuak irakurtzeko)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
# Aldatu zure Drive-ko karpetara. Bertan egon behar dute txartelak_train.jsonl eta txartelak_test.jsonl
KARPETA = "/content/drive/MyDrive/llm-egokitzapena"
os.makedirs(KARPETA, exist_ok=True)
TRAIN_FITX = f"{KARPETA}/txartelak_train.jsonl"
TEST_FITX = f"{KARPETA}/txartelak_test.jsonl"

# Fitxategiak Drive-n ez badaude, igo eskuz
if not os.path.exists(TRAIN_FITX):
    from google.colab import files  # COLAB
    igoak = files.upload()
    for izena in igoak:
        os.replace(izena, f"{KARPETA}/{izena}")
print(os.listdir(KARPETA))

## 2. Eredua kargatu eta harekin hitz egin

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELOA = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODELOA)
model = AutoModelForCausalLM.from_pretrained(MODELOA, dtype="auto", device_map="auto")
print(f"Parametroak: {model.num_parameters() / 1e6:.0f} M")
print(f"Memoria: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
def galdetu(mezuak, max_new_tokens=150):
    sarrera = tokenizer.apply_chat_template(mezuak, add_generation_prompt=True,
                                            return_tensors="pt", return_dict=True).to(model.device)
    irteera = model.generate(**sarrera, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(irteera[0][sarrera["input_ids"].shape[1]:], skip_special_tokens=True)

print(galdetu([{"role": "user", "content": "Zer da Python? Erantzun esaldi batean."}]))

In [ ]:
# Probatu galdera desberdinak: euskaraz, gaztelaniaz, ingelesez, kalkuluak...
for galdera in ["¿Qué es Python? Responde en una frase.",
                "What is Python? Answer in one sentence.",
                "Zenbat da 17 * 23?",
                "Idatzi Python funtzio bat zerrenda bat alderantzikatzeko."]:
    print(">>>", galdera)
    print(galdetu([{"role": "user", "content": galdera}]), "\n")

> **Galdera**: zer hizkuntzatan erantzuten du hobeto? Zuzenak dira kalkuluak? Idatzi zure behaketak hemen:
>
> ...

## 3. Datuak kargatu eta txat-txantiloia ikusi

In [ ]:
import json
def irakurri_jsonl(fitx):
    with open(fitx, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_lista = irakurri_jsonl(TRAIN_FITX)
test_lista = irakurri_jsonl(TEST_FITX)
SYSTEM = train_lista[0]["prompt"][0]["content"]   # sistema-mezua datuetatik (beti berdina!)
print(len(train_lista), len(test_lista))
print(SYSTEM)

In [ ]:
adib = test_lista[0]
print("SARRERA:", adib["prompt"][1]["content"])
print("ESPERO DEN IRTEERA:", adib["completion"][0]["content"])
print("\n--- Ereduak ikusten duen testua ---")
print(tokenizer.apply_chat_template(adib["prompt"], tokenize=False, add_generation_prompt=True))

## 4. Ebaluazio-funtzioak

In [ ]:
import json

def ebaluatu_bat(sortua, zuzena):
    """sortua: ereduaren testua; zuzena: dict-a (erantzun zuzena)."""
    em = {"json_ok": False, "kategoria": False, "lehentasuna": False,
          "kokapena": False, "guztiz": False}
    try:
        s = json.loads(sortua)
    except json.JSONDecodeError:
        return em
    if not isinstance(s, dict):
        return em
    em["json_ok"] = True
    for eremua in ["kategoria", "lehentasuna", "kokapena"]:
        em[eremua] = s.get(eremua) == zuzena[eremua]
    em["guztiz"] = em["kategoria"] and em["lehentasuna"] and em["kokapena"]
    return em

In [ ]:
def sortu_sortaka(mezu_zerrendak, model, tamaina=16, max_new_tokens=60):
    """Mezu-zerrenda askorentzat erantzunak sortu, sortaka eta deterministikoki."""
    tokenizer.padding_side = "left"           # sortzeko: betegarria ezkerrean
    emaitzak = []
    for i in range(0, len(mezu_zerrendak), tamaina):
        testuak = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                   for m in mezu_zerrendak[i:i + tamaina]]
        sarrera = tokenizer(testuak, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            irteera = model.generate(**sarrera, max_new_tokens=max_new_tokens, do_sample=False,
                                     pad_token_id=tokenizer.pad_token_id)
        berriak = irteera[:, sarrera["input_ids"].shape[1]:]
        emaitzak += tokenizer.batch_decode(berriak, skip_special_tokens=True)
    return emaitzak

In [ ]:
import pandas as pd

def ebaluatu_multzoa(adibideak, sortuak):
    lerroak = []
    for adib, sortua in zip(adibideak, sortuak):
        zuzena = json.loads(adib["completion"][0]["content"])
        lerroak.append({"testua": adib["prompt"][-1]["content"], "sortua": sortua,
                        "zuzena": adib["completion"][0]["content"],
                        "mota": adib.get("mota", ""), **ebaluatu_bat(sortua, zuzena)})
    return pd.DataFrame(lerroak)

METRIKAK = ["json_ok", "kategoria", "lehentasuna", "kokapena", "guztiz"]

In [ ]:
# Proba azkarra ebaluazio-funtzioarekin
print(ebaluatu_bat('{"kategoria": "sarea", "lehentasuna": "altua", "kokapena": null}',
                   {"kategoria": "sarea", "lehentasuna": "baxua", "kokapena": None}))
print(ebaluatu_bat('```json\n{"kategoria": "sarea"}\n```', {"kategoria": "sarea", "lehentasuna": "baxua", "kokapena": None}))

## 5. Oinarri-lerroa: zero-shot

Sistema-mezua + txartela bakarrik. Test multzoko lehen 50 adibideak erabiliko ditugu (azkarrago joateko).

In [ ]:
N = 50
lagina = test_lista[:N]
mezuak_zs = [a["prompt"] for a in lagina]       # [system, user]
sortuak_zs = sortu_sortaka(mezuak_zs, model)
df_zs = ebaluatu_multzoa(lagina, sortuak_zs)
print(df_zs[METRIKAK].mean().round(3))

In [ ]:
# Begiratu irteera batzuk
for _, r in df_zs.head(8).iterrows():
    print("TXARTELA:", r["testua"])
    print("SORTUA:  ", r["sortua"])
    print("ZUZENA:  ", r["zuzena"], "\n")

## 6. Oinarri-lerroa: few-shot

Train multzoko 4 adibide jarriko ditugu promptean, `user`/`assistant` mezu gisa.

In [ ]:
import random
random.seed(0)
# Adibide anitzak aukeratu: kategoria desberdinetakoak
ikusitakoak, adibideak = set(), []
for a in random.sample(train_lista, len(train_lista)):
    kat = json.loads(a["completion"][0]["content"])["kategoria"]
    if kat not in ikusitakoak:
        ikusitakoak.add(kat); adibideak.append(a)
    if len(adibideak) == 4:
        break

def few_shot_mezuak(adib, adibideak):
    mezuak = [{"role": "system", "content": SYSTEM}]
    for a in adibideak:
        mezuak += [a["prompt"][1], a["completion"][0]]   # user + assistant
    mezuak.append(adib["prompt"][1])
    return mezuak

for m in few_shot_mezuak(lagina[0], adibideak):
    print(m["role"], ":", m["content"][:90])

In [ ]:
mezuak_fs = [few_shot_mezuak(a, adibideak) for a in lagina]
sortuak_fs = sortu_sortaka(mezuak_fs, model, tamaina=8)
df_fs = ebaluatu_multzoa(lagina, sortuak_fs)
print(df_fs[METRIKAK].mean().round(3))

In [ ]:
# Zenbat token promptean? (kostua eta abiadura)
luz_zs = len(tokenizer.apply_chat_template(mezuak_zs[0], add_generation_prompt=True)["input_ids"])
luz_fs = len(tokenizer.apply_chat_template(mezuak_fs[0], add_generation_prompt=True)["input_ids"])
print(f"Zero-shot: {luz_zs} token | Few-shot: {luz_fs} token ({luz_fs / luz_zs:.1f}x)")

## 7. Emaitzak gorde

In [ ]:
laburpena = pd.DataFrame([
    {"eredua": "oinarria_zero_shot", "n": N, "prompt_tokenak": luz_zs, **df_zs[METRIKAK].mean().round(3).to_dict()},
    {"eredua": "oinarria_few_shot", "n": N, "prompt_tokenak": luz_fs, **df_fs[METRIKAK].mean().round(3).to_dict()},
])
laburpena.to_csv(f"{KARPETA}/oinarri_lerroa.csv", index=False)
laburpena

## 8. Ariketak

1. **Iragarpena**: saioaren hasieran zure iragarpena idatzi zenuen. Asmatu zenuen?
2. **JSON garbitzea**: idatzi `garbitu(testua)` funtzio bat, ```` ```json ```` blokeak eta JSON-aren aurreko/ondorengo testua kentzen dituena (aholkua: `re.search(r"\{.*\}", testua, re.S)`). Zenbat hobetzen da `json_ok`?
3. **Prompt engineering**: idatzi sistema-mezu hobea (adib. deklinabideari buruzko argibideak: *"kokapena nominatiboan: 'A12 gelan' → 'A12 gela'"*). Ebaluatu berriz zero-shot. Zein da lortzen duzun emaitzarik onena?
4. **Eredu handiagoa** (aukerakoa): probatu `Qwen/Qwen2.5-1.5B-Instruct` zero-shot. Zenbat hobetzen da? Zenbat memoria behar du?
5. Zein eremutan huts egiten du gehien oinarrizko ereduak? Zergatik uste duzu?

In [ ]:
# 2. ariketa
import re
def garbitu(testua):
    # ZURE KODEA HEMEN
    return testua